# SQL window functions: practice notebook

5 exercises, Easy to Hard, each with hidden tests.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("s1")` prints the reference solution for that exercise.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, pandas, scipy, scikit-learn and matplotlib)
import importlib.util, subprocess, sys
for _p in []:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The datasets every exercise uses. Read them if you like; don't change them.

In [ ]:
import sqlite3, random
from datetime import date, timedelta
_rr = random.Random(7)
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE employees(id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary INTEGER);
CREATE TABLE sales(day TEXT, region TEXT, revenue REAL);
CREATE TABLE users(user_id INTEGER PRIMARY KEY, signup_date TEXT);
CREATE TABLE activity(user_id INTEGER, day TEXT);
CREATE TABLE events(user_id INTEGER, event TEXT, ts TEXT);
""")
_names = ["Ava","Ben","Cleo","Dan","Eli","Fay","Gus","Hana","Ian","Jo","Kai","Lea","Max","Nia","Oto","Pia","Quin","Rae","Sam","Tia","Ugo","Vera","Wes","Xia"]
_rows = []
for i, n in enumerate(_names):
    _rows.append((i + 1, n, ["Quant", "Data", "Eng", "Ops"][i % 4], _rr.choice([90, 95, 100, 110, 120, 120, 130, 150]) * 1000))
con.executemany("INSERT INTO employees VALUES (?,?,?,?)", _rows)
_d0 = date(2026, 1, 1)
con.executemany("INSERT INTO sales VALUES (?,?,?)", [((_d0 + timedelta(days=d)).isoformat(), r, round(_rr.uniform(800, 1500) * (1 + 0.004 * d) * {"North": 1.2, "South": 1.0, "West": 0.8}[r], 2)) for d in range(90) for r in ("North", "South", "West")])
_u = []
for uid in range(1, 301):
    sd = _d0 + timedelta(days=_rr.randrange(0, 90))
    _u.append((uid, sd.isoformat()))
    k = _rr.random()
    for off in range(0, 120, 3):
        if _rr.random() < (0.9 if off < 7 else 0.35 * k):
            con.execute("INSERT INTO activity VALUES (?,?)", (uid, (sd + timedelta(days=off + _rr.randrange(3))).isoformat()))
con.executemany("INSERT INTO users VALUES (?,?)", _u)
_steps = ["visit", "add_to_cart", "checkout", "purchase"]
for uid in range(1, 1001):
    t = _d0 + timedelta(days=_rr.randrange(60))
    for s, pc in zip(_steps, (1.0, 0.42, 0.55, 0.7)):
        if _rr.random() > pc: break
        con.execute("INSERT INTO events VALUES (?,?,?)", (uid, s, t.isoformat()))
        t += timedelta(hours=1)
con.commit()
del _rr
def _q(sql):
    cur = con.execute(sql)
    return [d[0] for d in cur.description], cur.fetchall()

import pandas as pd
def sql(q):
    """Run a query and show the result as a table."""
    return pd.read_sql_query(q, con)


In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJzMSI6IHsic29sdXRpb24iOiAiV0lUSCByIEFTIChcbiAgU0VMRUNUIGRlcHQsIG5hbWUsIHNhbGFyeSxcbiAgICAgICAgIERFTlNFX1JBTksoKSBPVkVSIChQQVJUSVRJT04gQlkgZGVwdCBPUkRFUiBCWSBzYWxhcnkgREVTQykgQVMgcm5rXG4gIEZST00gZW1wbG95ZWVzXG4pXG5TRUxFQ1QgZGVwdCwgbmFtZSwgc2FsYXJ5LCBybmtcbkZST00gclxuV0hFUkUgcm5rIDw9IDJcbk9SREVSIEJZIGRlcHQsIHJuaywgbmFtZTtcbiIsICJ0ZXN0cyI6ICJcbmRlZiBfY21wKG9yZGVyPVRydWUsIGxhYmVsPVwiXCIpOlxuICAgIHVjLCB1ciA9IF9xKFVbXCJfX3NxbF9fXCJdKTsgcmMsIHJyID0gX3EoUltcIl9fc3FsX19cIl0pXG4gICAgYXNzZXJ0IFtjLmxvd2VyKCkgZm9yIGMgaW4gdWNdID09IFtjLmxvd2VyKCkgZm9yIGMgaW4gcmNdLCBmXCJjb2x1bW5zIGFyZSB7dWN9LCBleHBlY3RlZCB7cmN9XCJcbiAgICBhc3NlcnQgbGVuKHVyKSA9PSBsZW4ocnIpLCBmXCJ7bGVuKHVyKX0gcm93cywgZXhwZWN0ZWQge2xlbihycil9XCJcbiAgICBrZXkgPSBsYW1iZGEgcjogdHVwbGUoKHggaXMgTm9uZSwgcm91bmQoeCwgNikgaWYgaXNpbnN0YW5jZSh4LCBmbG9hdCkgZWxzZSB4KSBmb3IgeCBpbiByKVxuICAgIGEgPSBba2V5KHIpIGZvciByIGluIHVyXTsgYiA9IFtrZXkocikgZm9yIHIgaW4gcnJdXG4gICAgaWYgbm90IG9yZGVyOiBhLCBiID0gc29ydGVkKGEsIGtleT1zdHIpLCBzb3J0ZWQoYiwga2V5PXN0cilcbiAgICBmb3IgaSwgKHgsIHkpIGluIGVudW1lcmF0ZSh6aXAoYSwgYikpOlxuICAgICAgICBhc3NlcnQgeCA9PSB5LCBmXCJ7bGFiZWx9cm93IHtpICsgMX0gaXMge3VyW2ldIGlmIG9yZGVyIGVsc2UgeH0sIGV4cGVjdGVkIHtycltpXSBpZiBvcmRlciBlbHNlIHl9XCIgKyAoXCJcIiBpZiBvcmRlciBlbHNlIFwiIChzb3J0ZWQpXCIpXG5cbmRlZiBfaGlkZGVuKCk6XG4gICAgY29uLmV4ZWN1dGUoXCJJTlNFUlQgSU5UTyBlbXBsb3llZXMgVkFMVUVTICg5OSwgJ1plZCcsICdEYXRhJywgOTk5MDAwKSwgKDk4LCAnWWFuJywgJ0RhdGEnLCA5OTkwMDApXCIpXG4gICAgdHJ5OiBfY21wKGxhYmVsPVwid2l0aCB0d28gbmV3IHRpZWQgdG9wIGVhcm5lcnM6IFwiKVxuICAgIGZpbmFsbHk6IGNvbi5leGVjdXRlKFwiREVMRVRFIEZST00gZW1wbG95ZWVzIFdIRVJFIGlkIElOICg5OCwgOTkpXCIpXG5URVNUUyA9IFsoXCJzYW1wbGUgZGF0YVwiLCBsYW1iZGE6IF9jbXAoKSksIChcImhpZGRlbjogdGllcyBhdCB0aGUgdG9wXCIsIF9oaWRkZW4pXSIsICJsYW5nIjogInNxbCJ9LCAiczIiOiB7InNvbHV0aW9uIjogIldJVEggbSBBUyAoXG4gIFNFTEVDVCByZWdpb24sIHN0cmZ0aW1lKCclWS0lbScsIGRheSkgQVMgbW9udGgsIFNVTShyZXZlbnVlKSBBUyByZXZcbiAgRlJPTSBzYWxlc1xuICBHUk9VUCBCWSByZWdpb24sIG1vbnRoXG4pLCBsIEFTIChcbiAgU0VMRUNUIHJlZ2lvbiwgbW9udGgsIHJldiwgTEFHKHJldikgT1ZFUiAoUEFSVElUSU9OIEJZIHJlZ2lvbiBPUkRFUiBCWSBtb250aCkgQVMgcHJldlxuICBGUk9NIG1cbilcblNFTEVDVCByZWdpb24sIG1vbnRoLCBST1VORChyZXYsIDIpIEFTIHJldmVudWUsIFJPVU5EKHByZXYsIDIpIEFTIHByZXZfcmV2ZW51ZSxcbiAgICAgICBST1VORCgxMDAuMCAqIChyZXYgLSBwcmV2KSAvIHByZXYsIDIpIEFTIHBjdF9jaGFuZ2VcbkZST00gbFxuT1JERVIgQlkgcmVnaW9uLCBtb250aDtcbiIsICJ0ZXN0cyI6ICJcbmRlZiBfY21wKG9yZGVyPVRydWUsIGxhYmVsPVwiXCIpOlxuICAgIHVjLCB1ciA9IF9xKFVbXCJfX3NxbF9fXCJdKTsgcmMsIHJyID0gX3EoUltcIl9fc3FsX19cIl0pXG4gICAgYXNzZXJ0IFtjLmxvd2VyKCkgZm9yIGMgaW4gdWNdID09IFtjLmxvd2VyKCkgZm9yIGMgaW4gcmNdLCBmXCJjb2x1bW5zIGFyZSB7dWN9LCBleHBlY3RlZCB7cmN9XCJcbiAgICBhc3NlcnQgbGVuKHVyKSA9PSBsZW4ocnIpLCBmXCJ7bGVuKHVyKX0gcm93cywgZXhwZWN0ZWQge2xlbihycil9XCJcbiAgICBrZXkgPSBsYW1iZGEgcjogdHVwbGUoKHggaXMgTm9uZSwgcm91bmQoeCwgNikgaWYgaXNpbnN0YW5jZSh4LCBmbG9hdCkgZWxzZSB4KSBmb3IgeCBpbiByKVxuICAgIGEgPSBba2V5KHIpIGZvciByIGluIHVyXTsgYiA9IFtrZXkocikgZm9yIHIgaW4gcnJdXG4gICAgaWYgbm90IG9yZGVyOiBhLCBiID0gc29ydGVkKGEsIGtleT1zdHIpLCBzb3J0ZWQoYiwga2V5PXN0cilcbiAgICBmb3IgaSwgKHgsIHkpIGluIGVudW1lcmF0ZSh6aXAoYSwgYikpOlxuICAgICAgICBhc3NlcnQgeCA9PSB5LCBmXCJ7bGFiZWx9cm93IHtpICsgMX0gaXMge3VyW2ldIGlmIG9yZGVyIGVsc2UgeH0sIGV4cGVjdGVkIHtycltpXSBpZiBvcmRlciBlbHNlIHl9XCIgKyAoXCJcIiBpZiBvcmRlciBlbHNlIFwiIChzb3J0ZWQpXCIpXG5cblRFU1RTID0gWyhcInNhbXBsZSBkYXRhXCIsIGxhbWJkYTogX2NtcCgpKV0iLCAibGFuZyI6ICJzcWwifSwgInMzIjogeyJzb2x1dGlvbiI6ICJTRUxFQ1QgcmVnaW9uLCBkYXksIHJldmVudWUsXG4gICAgICAgUk9VTkQoU1VNKHJldmVudWUpIE9WRVIgKFBBUlRJVElPTiBCWSByZWdpb24gT1JERVIgQlkgZGF5IFJPV1MgQkVUV0VFTiBVTkJPVU5ERUQgUFJFQ0VESU5HIEFORCBDVVJSRU5UIFJPVyksIDIpIEFTIHJ1bm5pbmdfdG90YWwsXG4gICAgICAgUk9VTkQoQVZHKHJldmVudWUpIE9WRVIgKFBBUlRJVElPTiBCWSByZWdpb24gT1JERVIgQlkgZGF5IFJPV1MgQkVUV0VFTiA2IFBSRUNFRElORyBBTkQgQ1VSUkVOVCBST1cpLCAyKSBBUyBtYV83XG5GUk9NIHNhbGVzXG5PUkRFUiBCWSByZWdpb24sIGRheTtcbiIsICJ0ZXN0cyI6ICJcbmRlZiBfY21wKG9yZGVyPVRydWUsIGxhYmVsPVwiXCIpOlxuICAgIHVjLCB1ciA9IF9xKFVbXCJfX3NxbF9fXCJdKTsgcmMsIHJyID0gX3EoUltcIl9fc3FsX19cIl0pXG4gICAgYXNzZXJ0IFtjLmxvd2VyKCkgZm9yIGMgaW4gdWNdID09IFtjLmxvd2VyKCkgZm9yIGMgaW4gcmNdLCBmXCJjb2x1bW5zIGFyZSB7dWN9LCBleHBlY3RlZCB7cmN9XCJcbiAgICBhc3NlcnQgbGVuKHVyKSA9PSBsZW4ocnIpLCBmXCJ7bGVuKHVyKX0gcm93cywgZXhwZWN0ZWQge2xlbihycil9XCJcbiAgICBrZXkgPSBsYW1iZGEgcjogdHVwbGUoKHggaXMgTm9uZSwgcm91bmQoeCwgNikgaWYgaXNpbnN0YW5jZSh4LCBmbG9hdCkgZWxzZSB4KSBmb3IgeCBpbiByKVxuICAgIGEgPSBba2V5KHIpIGZvciByIGluIHVyXTsgYiA9IFtrZXkocikgZm9yIHIgaW4gcnJdXG4gICAgaWYgbm90IG9yZGVyOiBhLCBiID0gc29ydGVkKGEsIGtleT1zdHIpLCBzb3J0ZWQoYiwga2V5PXN0cilcbiAgICBmb3IgaSwgKHgsIHkpIGluIGVudW1lcmF0ZSh6aXAoYSwgYikpOlxuICAgICAgICBhc3NlcnQgeCA9PSB5LCBmXCJ7bGFiZWx9cm93IHtpICsgMX0gaXMge3VyW2ldIGlmIG9yZGVyIGVsc2UgeH0sIGV4cGVjdGVkIHtycltpXSBpZiBvcmRlciBlbHNlIHl9XCIgKyAoXCJcIiBpZiBvcmRlciBlbHNlIFwiIChzb3J0ZWQpXCIpXG5cblRFU1RTID0gWyhcInNhbXBsZSBkYXRhXCIsIGxhbWJkYTogX2NtcCgpKV0iLCAibGFuZyI6ICJzcWwifSwgInM0IjogeyJzb2x1dGlvbiI6ICJXSVRIIGMgQVMgKFxuICBTRUxFQ1QgZXZlbnQgQVMgc3RlcCwgQ09VTlQoRElTVElOQ1QgdXNlcl9pZCkgQVMgdXNlcnMsXG4gICAgICAgICBDQVNFIGV2ZW50IFdIRU4gJ3Zpc2l0JyBUSEVOIDEgV0hFTiAnYWRkX3RvX2NhcnQnIFRIRU4gMiBXSEVOICdjaGVja291dCcgVEhFTiAzIFdIRU4gJ3B1cmNoYXNlJyBUSEVOIDQgRU5EIEFTIHBvc1xuICBGUk9NIGV2ZW50c1xuICBHUk9VUCBCWSBldmVudFxuKVxuU0VMRUNUIHN0ZXAsIHVzZXJzLCBST1VORCgxMDAuMCAqIHVzZXJzIC8gTEFHKHVzZXJzKSBPVkVSIChPUkRFUiBCWSBwb3MpLCAxKSBBUyBwY3Rfb2ZfcHJldlxuRlJPTSBjXG5PUkRFUiBCWSBwb3M7XG4iLCAidGVzdHMiOiAiXG5kZWYgX2NtcChvcmRlcj1UcnVlLCBsYWJlbD1cIlwiKTpcbiAgICB1YywgdXIgPSBfcShVW1wiX19zcWxfX1wiXSk7IHJjLCByciA9IF9xKFJbXCJfX3NxbF9fXCJdKVxuICAgIGFzc2VydCBbYy5sb3dlcigpIGZvciBjIGluIHVjXSA9PSBbYy5sb3dlcigpIGZvciBjIGluIHJjXSwgZlwiY29sdW1ucyBhcmUge3VjfSwgZXhwZWN0ZWQge3JjfVwiXG4gICAgYXNzZXJ0IGxlbih1cikgPT0gbGVuKHJyKSwgZlwie2xlbih1cil9IHJvd3MsIGV4cGVjdGVkIHtsZW4ocnIpfVwiXG4gICAga2V5ID0gbGFtYmRhIHI6IHR1cGxlKCh4IGlzIE5vbmUsIHJvdW5kKHgsIDYpIGlmIGlzaW5zdGFuY2UoeCwgZmxvYXQpIGVsc2UgeCkgZm9yIHggaW4gcilcbiAgICBhID0gW2tleShyKSBmb3IgciBpbiB1cl07IGIgPSBba2V5KHIpIGZvciByIGluIHJyXVxuICAgIGlmIG5vdCBvcmRlcjogYSwgYiA9IHNvcnRlZChhLCBrZXk9c3RyKSwgc29ydGVkKGIsIGtleT1zdHIpXG4gICAgZm9yIGksICh4LCB5KSBpbiBlbnVtZXJhdGUoemlwKGEsIGIpKTpcbiAgICAgICAgYXNzZXJ0IHggPT0geSwgZlwie2xhYmVsfXJvdyB7aSArIDF9IGlzIHt1cltpXSBpZiBvcmRlciBlbHNlIHh9LCBleHBlY3RlZCB7cnJbaV0gaWYgb3JkZXIgZWxzZSB5fVwiICsgKFwiXCIgaWYgb3JkZXIgZWxzZSBcIiAoc29ydGVkKVwiKVxuXG5kZWYgX2hpZGRlbigpOlxuICAgIGNvbi5leGVjdXRlKFwiSU5TRVJUIElOVE8gZXZlbnRzIFNFTEVDVCB1c2VyX2lkLCBldmVudCwgdHMgRlJPTSBldmVudHMgV0hFUkUgdXNlcl9pZCA8PSA1MFwiKVxuICAgIHRyeTogX2NtcChsYWJlbD1cIndpdGggZHVwbGljYXRlZCBldmVudHM6IFwiKVxuICAgIGZpbmFsbHk6XG4gICAgICAgIGNvbi5leGVjdXRlKFwiREVMRVRFIEZST00gZXZlbnRzIFdIRVJFIHJvd2lkIElOIChTRUxFQ1Qgcm93aWQgRlJPTSBldmVudHMgV0hFUkUgdXNlcl9pZCA8PSA1MCBPUkRFUiBCWSByb3dpZCBERVNDIExJTUlUIChTRUxFQ1QgQ09VTlQoKikgLyAyIEZST00gZXZlbnRzIFdIRVJFIHVzZXJfaWQgPD0gNTApKVwiKVxuVEVTVFMgPSBbKFwic2FtcGxlIGRhdGFcIiwgbGFtYmRhOiBfY21wKCkpLCAoXCJoaWRkZW46IGR1cGxpY2F0ZSBldmVudHNcIiwgX2hpZGRlbildIiwgImxhbmciOiAic3FsIn0sICJzNSI6IHsic29sdXRpb24iOiAiV0lUSCB1IEFTIChcbiAgU0VMRUNUIHVzZXJfaWQsIHN0cmZ0aW1lKCclWS0lbScsIHNpZ251cF9kYXRlKSBBUyBjb2hvcnQsXG4gICAgICAgICBDQVNUKHN0cmZ0aW1lKCclWScsIHNpZ251cF9kYXRlKSBBUyBJTlQpICogMTIgKyBDQVNUKHN0cmZ0aW1lKCclbScsIHNpZ251cF9kYXRlKSBBUyBJTlQpIEFTIG0wXG4gIEZST00gdXNlcnNcbiksIGEgQVMgKFxuICBTRUxFQ1QgRElTVElOQ1QgdS51c2VyX2lkLCB1LmNvaG9ydCxcbiAgICAgICAgIENBU1Qoc3RyZnRpbWUoJyVZJywgYWMuZGF5KSBBUyBJTlQpICogMTIgKyBDQVNUKHN0cmZ0aW1lKCclbScsIGFjLmRheSkgQVMgSU5UKSAtIHUubTAgQVMga1xuICBGUk9NIGFjdGl2aXR5IGFjIEpPSU4gdSBPTiB1LnVzZXJfaWQgPSBhYy51c2VyX2lkXG4pLCBzIEFTIChcbiAgU0VMRUNUIGNvaG9ydCwgQ09VTlQoKikgQVMgY29ob3J0X3NpemUgRlJPTSB1IEdST1VQIEJZIGNvaG9ydFxuKVxuU0VMRUNUIHMuY29ob3J0LCBzLmNvaG9ydF9zaXplLFxuICAgICAgIENPVU5UKERJU1RJTkNUIENBU0UgV0hFTiBhLmsgPSAxIFRIRU4gYS51c2VyX2lkIEVORCkgQVMgbTEsXG4gICAgICAgUk9VTkQoMTAwLjAgKiBDT1VOVChESVNUSU5DVCBDQVNFIFdIRU4gYS5rID0gMSBUSEVOIGEudXNlcl9pZCBFTkQpIC8gcy5jb2hvcnRfc2l6ZSwgMSkgQVMgbTFfcGN0LFxuICAgICAgIENPVU5UKERJU1RJTkNUIENBU0UgV0hFTiBhLmsgPSAyIFRIRU4gYS51c2VyX2lkIEVORCkgQVMgbTIsXG4gICAgICAgUk9VTkQoMTAwLjAgKiBDT1VOVChESVNUSU5DVCBDQVNFIFdIRU4gYS5rID0gMiBUSEVOIGEudXNlcl9pZCBFTkQpIC8gcy5jb2hvcnRfc2l6ZSwgMSkgQVMgbTJfcGN0XG5GUk9NIHMgTEVGVCBKT0lOIGEgT04gYS5jb2hvcnQgPSBzLmNvaG9ydFxuR1JPVVAgQlkgcy5jb2hvcnQsIHMuY29ob3J0X3NpemVcbk9SREVSIEJZIHMuY29ob3J0O1xuIiwgInRlc3RzIjogIlxuZGVmIF9jbXAob3JkZXI9VHJ1ZSwgbGFiZWw9XCJcIik6XG4gICAgdWMsIHVyID0gX3EoVVtcIl9fc3FsX19cIl0pOyByYywgcnIgPSBfcShSW1wiX19zcWxfX1wiXSlcbiAgICBhc3NlcnQgW2MubG93ZXIoKSBmb3IgYyBpbiB1Y10gPT0gW2MubG93ZXIoKSBmb3IgYyBpbiByY10sIGZcImNvbHVtbnMgYXJlIHt1Y30sIGV4cGVjdGVkIHtyY31cIlxuICAgIGFzc2VydCBsZW4odXIpID09IGxlbihyciksIGZcIntsZW4odXIpfSByb3dzLCBleHBlY3RlZCB7bGVuKHJyKX1cIlxuICAgIGtleSA9IGxhbWJkYSByOiB0dXBsZSgoeCBpcyBOb25lLCByb3VuZCh4LCA2KSBpZiBpc2luc3RhbmNlKHgsIGZsb2F0KSBlbHNlIHgpIGZvciB4IGluIHIpXG4gICAgYSA9IFtrZXkocikgZm9yIHIgaW4gdXJdOyBiID0gW2tleShyKSBmb3IgciBpbiBycl1cbiAgICBpZiBub3Qgb3JkZXI6IGEsIGIgPSBzb3J0ZWQoYSwga2V5PXN0ciksIHNvcnRlZChiLCBrZXk9c3RyKVxuICAgIGZvciBpLCAoeCwgeSkgaW4gZW51bWVyYXRlKHppcChhLCBiKSk6XG4gICAgICAgIGFzc2VydCB4ID09IHksIGZcIntsYWJlbH1yb3cge2kgKyAxfSBpcyB7dXJbaV0gaWYgb3JkZXIgZWxzZSB4fSwgZXhwZWN0ZWQge3JyW2ldIGlmIG9yZGVyIGVsc2UgeX1cIiArIChcIlwiIGlmIG9yZGVyIGVsc2UgXCIgKHNvcnRlZClcIilcblxuVEVTVFMgPSBbKFwic2FtcGxlIGRhdGFcIiwgbGFtYmRhOiBfY21wKCkpXSIsICJsYW5nIjogInNxbCJ9fQ==").decode())
_SETUP = _b64.b64decode("CmltcG9ydCBzcWxpdGUzLCByYW5kb20KZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZSwgdGltZWRlbHRhCl9yciA9IHJhbmRvbS5SYW5kb20oNykKY29uID0gc3FsaXRlMy5jb25uZWN0KCI6bWVtb3J5OiIpCmNvbi5leGVjdXRlc2NyaXB0KCIiIgpDUkVBVEUgVEFCTEUgZW1wbG95ZWVzKGlkIElOVEVHRVIgUFJJTUFSWSBLRVksIG5hbWUgVEVYVCwgZGVwdCBURVhULCBzYWxhcnkgSU5URUdFUik7CkNSRUFURSBUQUJMRSBzYWxlcyhkYXkgVEVYVCwgcmVnaW9uIFRFWFQsIHJldmVudWUgUkVBTCk7CkNSRUFURSBUQUJMRSB1c2Vycyh1c2VyX2lkIElOVEVHRVIgUFJJTUFSWSBLRVksIHNpZ251cF9kYXRlIFRFWFQpOwpDUkVBVEUgVEFCTEUgYWN0aXZpdHkodXNlcl9pZCBJTlRFR0VSLCBkYXkgVEVYVCk7CkNSRUFURSBUQUJMRSBldmVudHModXNlcl9pZCBJTlRFR0VSLCBldmVudCBURVhULCB0cyBURVhUKTsKIiIiKQpfbmFtZXMgPSBbIkF2YSIsIkJlbiIsIkNsZW8iLCJEYW4iLCJFbGkiLCJGYXkiLCJHdXMiLCJIYW5hIiwiSWFuIiwiSm8iLCJLYWkiLCJMZWEiLCJNYXgiLCJOaWEiLCJPdG8iLCJQaWEiLCJRdWluIiwiUmFlIiwiU2FtIiwiVGlhIiwiVWdvIiwiVmVyYSIsIldlcyIsIlhpYSJdCl9yb3dzID0gW10KZm9yIGksIG4gaW4gZW51bWVyYXRlKF9uYW1lcyk6CiAgICBfcm93cy5hcHBlbmQoKGkgKyAxLCBuLCBbIlF1YW50IiwgIkRhdGEiLCAiRW5nIiwgIk9wcyJdW2kgJSA0XSwgX3JyLmNob2ljZShbOTAsIDk1LCAxMDAsIDExMCwgMTIwLCAxMjAsIDEzMCwgMTUwXSkgKiAxMDAwKSkKY29uLmV4ZWN1dGVtYW55KCJJTlNFUlQgSU5UTyBlbXBsb3llZXMgVkFMVUVTICg/LD8sPyw/KSIsIF9yb3dzKQpfZDAgPSBkYXRlKDIwMjYsIDEsIDEpCmNvbi5leGVjdXRlbWFueSgiSU5TRVJUIElOVE8gc2FsZXMgVkFMVUVTICg/LD8sPykiLCBbKChfZDAgKyB0aW1lZGVsdGEoZGF5cz1kKSkuaXNvZm9ybWF0KCksIHIsIHJvdW5kKF9yci51bmlmb3JtKDgwMCwgMTUwMCkgKiAoMSArIDAuMDA0ICogZCkgKiB7Ik5vcnRoIjogMS4yLCAiU291dGgiOiAxLjAsICJXZXN0IjogMC44fVtyXSwgMikpIGZvciBkIGluIHJhbmdlKDkwKSBmb3IgciBpbiAoIk5vcnRoIiwgIlNvdXRoIiwgIldlc3QiKV0pCl91ID0gW10KZm9yIHVpZCBpbiByYW5nZSgxLCAzMDEpOgogICAgc2QgPSBfZDAgKyB0aW1lZGVsdGEoZGF5cz1fcnIucmFuZHJhbmdlKDAsIDkwKSkKICAgIF91LmFwcGVuZCgodWlkLCBzZC5pc29mb3JtYXQoKSkpCiAgICBrID0gX3JyLnJhbmRvbSgpCiAgICBmb3Igb2ZmIGluIHJhbmdlKDAsIDEyMCwgMyk6CiAgICAgICAgaWYgX3JyLnJhbmRvbSgpIDwgKDAuOSBpZiBvZmYgPCA3IGVsc2UgMC4zNSAqIGspOgogICAgICAgICAgICBjb24uZXhlY3V0ZSgiSU5TRVJUIElOVE8gYWN0aXZpdHkgVkFMVUVTICg/LD8pIiwgKHVpZCwgKHNkICsgdGltZWRlbHRhKGRheXM9b2ZmICsgX3JyLnJhbmRyYW5nZSgzKSkpLmlzb2Zvcm1hdCgpKSkKY29uLmV4ZWN1dGVtYW55KCJJTlNFUlQgSU5UTyB1c2VycyBWQUxVRVMgKD8sPykiLCBfdSkKX3N0ZXBzID0gWyJ2aXNpdCIsICJhZGRfdG9fY2FydCIsICJjaGVja291dCIsICJwdXJjaGFzZSJdCmZvciB1aWQgaW4gcmFuZ2UoMSwgMTAwMSk6CiAgICB0ID0gX2QwICsgdGltZWRlbHRhKGRheXM9X3JyLnJhbmRyYW5nZSg2MCkpCiAgICBmb3IgcywgcGMgaW4gemlwKF9zdGVwcywgKDEuMCwgMC40MiwgMC41NSwgMC43KSk6CiAgICAgICAgaWYgX3JyLnJhbmRvbSgpID4gcGM6IGJyZWFrCiAgICAgICAgY29uLmV4ZWN1dGUoIklOU0VSVCBJTlRPIGV2ZW50cyBWQUxVRVMgKD8sPyw/KSIsICh1aWQsIHMsIHQuaXNvZm9ybWF0KCkpKQogICAgICAgIHQgKz0gdGltZWRlbHRhKGhvdXJzPTEpCmNvbi5jb21taXQoKQpkZWwgX3JyCmRlZiBfcShzcWwpOgogICAgY3VyID0gY29uLmV4ZWN1dGUoc3FsKQogICAgcmV0dXJuIFtkWzBdIGZvciBkIGluIGN1ci5kZXNjcmlwdGlvbl0sIGN1ci5mZXRjaGFsbCgpCg==").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Top earners per department
*Easy · id `s1`*

<p>Table <code>employees(id, name, dept, salary)</code>.</p>
<p>For each department return its top 2 salary levels: columns <code>dept, name, salary, rnk</code>, where <code>rnk</code> is the <code>DENSE_RANK</code> of salary within the department, highest first. Keep everyone with rnk ≤ 2 (ties share a rank). Order by <code>dept, rnk, name</code>.</p>
<p class=hint>Window functions can't go in WHERE: rank in a CTE or subquery, then filter.</p>

In [ ]:
q_s1 = """
-- write one SQLite query
SELECT *
FROM employees
"""
sql(q_s1)

In [ ]:
check("s1", q_s1)

---
## 2. Month-over-month revenue
*Medium · id `s2`*

<p>Table <code>sales(day, region, revenue)</code>, one row per region per day, <code>day</code> as <code>'YYYY-MM-DD'</code> text.</p>
<p>Return <code>region, month, revenue, prev_revenue, pct_change</code>: monthly revenue per region (month as <code>'YYYY-MM'</code>, revenue rounded to 2), the previous month's revenue for the same region (rounded to 2, NULL for the first month), and <code>ROUND(100.0 * (revenue - prev_revenue) / prev_revenue, 2)</code> computed from the unrounded sums. Order by <code>region, month</code>.</p>
<p class=hint><code>strftime('%Y-%m', day)</code>, then <code>LAG(...) OVER (PARTITION BY region ORDER BY month)</code>.</p>

In [ ]:
q_s2 = """
SELECT *
FROM sales
LIMIT 10;
"""
sql(q_s2)

In [ ]:
check("s2", q_s2)

---
## 3. Running total and 7-day moving average
*Medium · id `s3`*

<p>Using <code>sales</code>, return <code>region, day, revenue, running_total, ma_7</code>: the running total of revenue per region up to and including the day (rounded to 2), and the average over the current and previous 6 rows of that region (rounded to 2; fewer rows at the start). Order by <code>region, day</code>.</p>
<p class=hint><code>ROWS BETWEEN 6 PRECEDING AND CURRENT ROW</code>. Say out loud why ROWS and RANGE differ when days are missing.</p>

In [ ]:
q_s3 = """
SELECT region, day, revenue
FROM sales
ORDER BY region, day;
"""
sql(q_s3)

In [ ]:
check("s3", q_s3)

---
## 4. Conversion funnel
*Medium · id `s4`*

<p>Table <code>events(user_id, event, ts)</code>. The funnel steps, in order, are <code>visit → add_to_cart → checkout → purchase</code>.</p>
<p>Return <code>step, users, pct_of_prev</code>: one row per step in funnel order, the number of distinct users with that event, and <code>ROUND(100.0 * users / previous step's users, 1)</code> (NULL for the first step).</p>
<p class=hint>Map each event to its position with <code>CASE</code>, count, then <code>LAG</code> over that position.</p>

In [ ]:
q_s4 = """
SELECT event, COUNT(*)
FROM events
GROUP BY event;
"""
sql(q_s4)

In [ ]:
check("s4", q_s4)

---
## 5. Monthly retention by signup cohort
*Hard · id `s5`*

<p>Tables <code>users(user_id, signup_date)</code> and <code>activity(user_id, day)</code>.</p>
<p>A user's cohort is their signup month (<code>'YYYY-MM'</code>). A user is retained in month k if they have any activity in the calendar month k months after signup. Return <code>cohort, cohort_size, m1, m1_pct, m2, m2_pct</code>: the number of users retained in months 1 and 2, and each as <code>ROUND(100.0 * m / cohort_size, 1)</code>. Order by <code>cohort</code>.</p>
<p class=hint>Month index = year × 12 + month: <code>CAST(strftime('%Y', d) AS INT) * 12 + CAST(strftime('%m', d) AS INT)</code>. Count DISTINCT users per offset, LEFT JOIN so cohorts with 0 keep a row.</p>

In [ ]:
q_s5 = """
SELECT *
FROM users
LIMIT 5;
"""
sql(q_s5)

In [ ]:
check("s5", q_s5)